In [ ]:
from pathlib import Path
import pandas as pd
train_data = pd.read_csv('dataset/train.csv')
test_data = pd.read_csv('dataset/test.csv')
train_death_data = pd.read_csv('dataset/gender_submission.csv')

print(train_data.head())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
IMAGES_PATH = Path()/"images"
IMAGES_PATH.mkdir(parents=True , exist_ok=True)
def save_fig(fig_id , tight_layout= True ,fig_extension="png" , resolution= 300):
    path = IMAGES_PATH /f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension,dpi=resolution)

In [ ]:
train_data['family_size'] = train_data['SibSp'] + train_data['Parch']

print(train_data.head())
print(train_data.info())


In [ ]:
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].mean())
print(train_data.info())    

In [ ]:
numeric_train_data = train_data.select_dtypes(include=[np.number])
print(numeric_train_data.head())

In [ ]:

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
itera_imputer = IterativeImputer(initial_strategy= 'median' , random_state= 42)
array_imputer = itera_imputer.fit_transform(numeric_train_data)

enhansed_numeric_train_data = pd.DataFrame(
    array_imputer,
    columns=numeric_train_data.columns ,
    index= numeric_train_data.index
    )

print(enhansed_numeric_train_data.head())


In [ ]:
new_train_data = train_data.drop(["Cabin" , "Ticket"] , axis= 1)

print(new_train_data.head())


In [ ]:
y_train = train_data["Survived"].copy()
x_train = new_train_data.drop(["Survived","Name"] ,axis = 1)
x_train.head()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
        
    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  
        
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
        
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
num_attibs = ["Pclass", "Age", "SibSp",  "Parch"  ,   "Fare" , "family_size"]
cat_attribute =[ "Sex" ,  "Embarked" ]
cluster_cluster_attribs = ["Age", "Fare"]

nums_pipeline = make_pipeline(IterativeImputer(initial_strategy="median", random_state= 42),StandardScaler())
cat_pipeline = make_pipeline( SimpleImputer(strategy="most_frequent"),OneHotEncoder(handle_unknown="ignore"))
cluster_pipline = make_pipeline(IterativeImputer(initial_strategy="median", random_state=42),
    StandardScaler(),
    ClusterSimilarity(n_clusters=5, gamma=1.0, random_state=42)
)

preprocessing = ColumnTransformer([
    ("num" , nums_pipeline ,num_attibs),
    ("cat" , cat_pipeline , cat_attribute),
    ("cluster" , cluster_pipline , cluster_cluster_attribs)
])
# Fix the categorical preprocessing: SimpleImputer handles strings; IterativeImputer does not.
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

preprocessing = ColumnTransformer([
    ("num", nums_pipeline, num_attibs),
    ("cat", cat_pipeline, cat_attribute),
    ("cluster", cluster_pipline, cluster_cluster_attribs),
])

X_train_prepared = preprocessing.fit_transform(x_train)

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000 , random_state= 42),
    "Random Forest":RandomForestClassifier(random_state= 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

for names, model in models.items():
    fullpipeline = Pipeline([
        ("preprocessing" , preprocessing),
        ("model" , model)
    ])

    scores = cross_val_score(fullpipeline , x_train , y_train , cv= 5 , scoring="accuracy")
    print(f"{names} -> Average Accurace : {scores.mean():.3f} (Standard Deviation: {scores.std():.3f})")
    


In [ ]:
test_data['family_size'] = test_data['SibSp'] + test_data['Parch']
X_test = test_data.drop(["Cabin", "Ticket", "Name", "PassengerId"], axis=1)
from sklearn.ensemble import GradientBoostingClassifier
final_pipeline = Pipeline([
    ("preprocessing" , preprocessing),
    ("Model" , GradientBoostingClassifier(random_state=42))
])
test_passenger_ids = test_data['PassengerId'].copy()
final_pipeline.fit(x_train , y_train)
prediction = final_pipeline.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test_passenger_ids,
    "Survived": prediction
})
final_score = cross_val_score(final_pipeline , x_train , y_train , cv = 5 , scoring="accuracy" )

submission.to_csv("titanic_submission.csv", index=False)

print("Predictions saved to titanic_submission.csv successfully!")
print(submission.head())